# Segmentation Clients – AnyCompany Food & Beverage

**Objectif** : Identifier des segments clients à fort potentiel via clustering (KMeans) basé sur le comportement démographique et les interactions.

**Données** : `ANYCOMPANY_LAB.ANALYTICS.CLIENTS_ENRICHIS`

In [ ]:
%%sql -r df_clients
SELECT 
    CUSTOMER_ID, GENDER, REGION, MARITAL_STATUS,
    ANNUAL_INCOME, AGE, TRANCHE_AGE, SEGMENT_REVENU,
    COALESCE(NB_INTERACTIONS, 0) AS NB_INTERACTIONS,
    COALESCE(SATISFACTION_MOYENNE, 3) AS SATISFACTION_MOYENNE,
    COALESCE(TAUX_RESOLUTION, 0) AS TAUX_RESOLUTION
FROM ANYCOMPANY_LAB.ANALYTICS.CLIENTS_ENRICHIS

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

print(f"Shape: {df_clients.shape}")
print(f"Colonnes: {list(df_clients.columns)}")
df_clients.head()

In [ ]:
features_num = ['ANNUAL_INCOME', 'AGE', 'NB_INTERACTIONS', 'SATISFACTION_MOYENNE', 'TAUX_RESOLUTION']

le_gender = LabelEncoder()
le_marital = LabelEncoder()
le_region = LabelEncoder()

df_ml = df_clients.copy()
df_ml['GENDER_ENC'] = le_gender.fit_transform(df_ml['GENDER'].astype(str))
df_ml['MARITAL_ENC'] = le_marital.fit_transform(df_ml['MARITAL_STATUS'].astype(str))
df_ml['REGION_ENC'] = le_region.fit_transform(df_ml['REGION'].astype(str))

feature_cols = features_num + ['GENDER_ENC', 'MARITAL_ENC', 'REGION_ENC']
X = df_ml[feature_cols].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Features: {feature_cols}")
print(f"Shape après scaling: {X_scaled.shape}")

In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(K_range, inertias, 'bo-')
ax1.set_xlabel('Nombre de clusters (K)')
ax1.set_ylabel('Inertie')
ax1.set_title('Methode du Coude')

ax2.plot(K_range, silhouettes, 'ro-')
ax2.set_xlabel('Nombre de clusters (K)')
ax2.set_ylabel('Score Silhouette')
ax2.set_title('Score Silhouette par K')

plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(silhouettes)]
print(f"Meilleur K (silhouette): {best_k} avec score={max(silhouettes):.4f}")

In [ ]:
K_OPTIMAL = 4

kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
df_ml['CLUSTER'] = kmeans.fit_predict(X_scaled)

print(f"Score silhouette final: {silhouette_score(X_scaled, df_ml['CLUSTER']):.4f}")
print(f"\nDistribution des clusters:")
print(df_ml['CLUSTER'].value_counts().sort_index())

In [ ]:
cluster_profile = df_ml.groupby('CLUSTER').agg({
    'ANNUAL_INCOME': ['mean', 'median'],
    'AGE': ['mean', 'median'],
    'NB_INTERACTIONS': 'mean',
    'SATISFACTION_MOYENNE': 'mean',
    'TAUX_RESOLUTION': 'mean',
    'CUSTOMER_ID': 'count'
}).round(2)

cluster_profile.columns = ['Revenu_Moyen', 'Revenu_Median', 'Age_Moyen', 'Age_Median',
                           'Interactions_Moy', 'Satisfaction_Moy', 'Resolution_Moy', 'Nb_Clients']
print(cluster_profile.to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']

for cluster_id in range(K_OPTIMAL):
    mask = df_ml['CLUSTER'] == cluster_id
    axes[0, 0].scatter(df_ml.loc[mask, 'ANNUAL_INCOME'], df_ml.loc[mask, 'AGE'],
                       c=colors[cluster_id], label=f'Cluster {cluster_id}', alpha=0.5)
axes[0, 0].set_xlabel('Revenu Annuel')
axes[0, 0].set_ylabel('Age')
axes[0, 0].set_title('Clusters: Revenu vs Age')
axes[0, 0].legend()

df_ml.groupby('CLUSTER')['ANNUAL_INCOME'].mean().plot(kind='bar', ax=axes[0, 1], color=colors)
axes[0, 1].set_title('Revenu moyen par cluster')
axes[0, 1].set_ylabel('Revenu moyen')

df_ml.groupby('CLUSTER')['AGE'].mean().plot(kind='bar', ax=axes[1, 0], color=colors)
axes[1, 0].set_title('Age moyen par cluster')
axes[1, 0].set_ylabel('Age moyen')

df_ml.groupby('CLUSTER')['SATISFACTION_MOYENNE'].mean().plot(kind='bar', ax=axes[1, 1], color=colors)
axes[1, 1].set_title('Satisfaction moyenne par cluster')
axes[1, 1].set_ylabel('Satisfaction')

plt.tight_layout()
plt.show()

In [ ]:
segment_names = {
    0: 'A definir apres analyse',
    1: 'A definir apres analyse',
    2: 'A definir apres analyse',
    3: 'A definir apres analyse'
}

print("=" * 60)
print("RECOMMANDATIONS MARKETING PAR SEGMENT")
print("=" * 60)
for cluster_id in range(K_OPTIMAL):
    mask = df_ml['CLUSTER'] == cluster_id
    subset = df_ml[mask]
    print(f"\n--- Cluster {cluster_id} ({len(subset)} clients) ---")
    print(f"  Revenu moyen: ${subset['ANNUAL_INCOME'].mean():,.0f}")
    print(f"  Age moyen: {subset['AGE'].mean():.0f} ans")
    print(f"  Satisfaction: {subset['SATISFACTION_MOYENNE'].mean():.2f}/5")
    top_region = subset['REGION'].value_counts().index[0]
    top_gender = subset['GENDER'].value_counts().index[0]
    print(f"  Region dominante: {top_region}")
    print(f"  Genre dominant: {top_gender}")

In [ ]:
%%sql -r ctx
USE DATABASE ANYCOMPANY_LAB;
USE SCHEMA ANALYTICS;